# 📈 Week 16 – Day 4: Build the First Stock Price Forecasting Model
## Linear Regression Stock Price Predictor

This notebook covers:
- Loading feature-engineered time series data (`processed_stock_data.csv`)
- Setting target column `Target = Close.shift(-1)` to predict next day closing price
- Time series sequential `train_test_split` without shuffling
- Training a Linear Regression model
- Evaluating performance using MAE, RMSE, and R² Score
- Visualizing Actual vs. Predicted values
- Forecasting tomorrow's stock price and saving the model artifact.

### 📥 Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use("ggplot")

### 📂 Step 2: Load Processed Dataset

In [2]:
stock = pd.read_csv("../data/processed_stock_data.csv", index_col="Date", parse_dates=True)
stock.head()

### 🎯 Step 3: Create the Target Column

We shift the Close price by `-1` so today's features predict tomorrow's closing price.

In [3]:
stock["Target"] = stock["Close"].shift(-1)
# Remove the last row which contains NaN as target
stock.dropna(inplace=True)
stock.tail()

### 📋 Step 4: Select Features

In [4]:
features = [
    "Open", "High", "Low", "Volume",
    "Lag_1", "Lag_2", "Lag_3", "Lag_5", "Lag_10",
    "MA10", "MA20", "MA50", "STD20",
    "Daily_Return", "Return_MA10", "Return_STD10"
]
X = stock[features]
y = stock["Target"]

### ✂️ Step 5: Train-Test Split

⚠️ **Important**: `shuffle=False` is preserved to keep chronological sequence.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

### 🤖 Step 6: Train the Model

In [6]:
model = LinearRegression()
model.fit(X_train, y_train)

### 📈 Step 7: Make Predictions

In [7]:
predictions = model.predict(X_test)
results = pd.DataFrame({"Actual": y_test, "Predicted": predictions})
results.head()

### 📊 Step 8: Evaluate the Model

In [8]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("MAE:     ", mae)
print("RMSE:    ", rmse)
print("R² Score:", r2)

### 📉 Step 9: Visualize Actual vs Predicted

In [9]:
plt.figure(figsize=(14, 6))
plt.plot(y_test.values, label="Actual")
plt.plot(predictions, label="Predicted")
plt.title("Actual vs Predicted Closing Price")
plt.xlabel("Days")
plt.ylabel("Price")
plt.legend()
plt.savefig("../images/actual_vs_predicted.png", dpi=300)
plt.show()

### 📌 Step 10: Predict Tomorrow's Closing Price

In [10]:
latest_data = stock[features].iloc[-1:]
tomorrow_price = model.predict(latest_data)
print("Predicted Next Closing Price:", tomorrow_price[0])

### 💾 Step 11: Save the Model

In [11]:
import os
os.makedirs("../model", exist_ok=True)
joblib.dump(model, "../model/linear_regression_stock.pkl")
print("Model saved successfully to ../model/linear_regression_stock.pkl")